# DSFB-DSCD results replay (zip upload + local outputs)

This notebook replays DSCD results generated by the Rust crate.

Generate local outputs (optional):

```bash
cargo run --release -p dsfb-dscd -- --num-events 2048 --tau-steps 201
```

Interactive dropdown controls for `num-events`, `tau-steps`, and graph preview edge count are provided in the first code cell.

Then either:
- place outputs under `output-dsfb-dscd/<timestamp>/`, or
- upload a `.zip` from your local machine (Colab picker supported).

Accepted zip layouts:
- `output-dsfb-dscd/<timestamp>/...`
- `<timestamp>/...` (contains either `threshold_sweep.csv` or `threshold_scaling_summary.csv` + `threshold_curve_N_*.csv`, plus `finite_size_scaling.csv`)



In [ ]:
from pathlib import Path\nfrom typing import List, Optional\nimport os\nimport shutil\nimport subprocess\nimport zipfile\nimport io\nimport csv\n\nAUTO_ENFORCE_LATEST_MAIN = False\nDSFB_MAIN_REPO = 'https://github.com/infinityabundance/dsfb.git'\n\ndef _run_git(repo_root: Path, *args: str) -> subprocess.CompletedProcess:\n    return subprocess.run(\n        ['git', *args],\n        cwd=repo_root,\n        stdout=subprocess.PIPE,\n        stderr=subprocess.PIPE,\n        text=True,\n    )\n\n\ndef _git_repo_root() -> Optional[Path]:\n    if shutil.which('git') is None:\n        return None\n    probe = subprocess.run(\n        ['git', 'rev-parse', '--show-toplevel'],\n        stdout=subprocess.PIPE,\n        stderr=subprocess.PIPE,\n        text=True,\n    )\n    if probe.returncode != 0:\n        return None\n    return Path(probe.stdout.strip()).resolve()\n\n\ndef _in_colab_runtime() -> bool:\n    return Path('/content').exists()\n\n\ndef ensure_latest_main() -> Path:\n    if not AUTO_ENFORCE_LATEST_MAIN:\n        repo_root = _git_repo_root()\n        return repo_root if repo_root is not None else Path.cwd().resolve()\n\n    repo_root = _git_repo_root()\n    if repo_root is None:\n        if not _in_colab_runtime():\n            raise RuntimeError(\n                'git repo not detected. Open this notebook from a dsfb git checkout of main.'\n            )\n\n        colab_repo = Path('/content/dsfb')\n        if colab_repo.exists() and not (colab_repo / '.git').exists():\n            raise RuntimeError(\n                '/content/dsfb exists but is not a git repository. Remove it and rerun.'\n            )\n        if not colab_repo.exists():\n            print(f'Cloning DSFB main into {colab_repo} ...')\n            clone = subprocess.run(\n                ['git', 'clone', '--depth', '1', '--branch', 'main', DSFB_MAIN_REPO, str(colab_repo)],\n                stdout=subprocess.PIPE,\n                stderr=subprocess.PIPE,\n                text=True,\n            )\n            if clone.returncode != 0:\n                raise RuntimeError(f'Failed to clone DSFB main:\n{clone.stderr.strip()}')\n        repo_root = colab_repo.resolve()\n        os.chdir(repo_root)\n        print(f'Using cloned repository at {repo_root}')\n\n    fetch = _run_git(repo_root, 'fetch', 'origin', 'main')\n    if fetch.returncode != 0:\n        raise RuntimeError(f'Failed to fetch origin/main:\n{fetch.stderr.strip()}')\n\n    local = _run_git(repo_root, 'rev-parse', 'HEAD')\n    remote = _run_git(repo_root, 'rev-parse', 'origin/main')\n    if local.returncode != 0 or remote.returncode != 0:\n        raise RuntimeError('Failed to resolve HEAD/origin/main SHA.')\n\n    local_sha = local.stdout.strip()\n    remote_sha = remote.stdout.strip()\n    print(f'Repo HEAD={local_sha[:12]} origin/main={remote_sha[:12]}')\n    if local_sha == remote_sha:\n        return repo_root\n\n    dirty = _run_git(repo_root, 'status', '--porcelain')\n    if dirty.returncode != 0:\n        print('Warning: repo status check failed; continuing with current checkout for replay mode.')\n        return repo_root\n    if dirty.stdout.strip():\n        print('Warning: repo is behind origin/main and has local changes; using current checkout for replay mode.')\n        return repo_root\n\n    checkout = _run_git(repo_root, 'checkout', 'main')\n    if checkout.returncode != 0:\n        raise RuntimeError(f'Failed to checkout main:\n{checkout.stderr.strip()}')\n    pull = _run_git(repo_root, 'pull', '--ff-only', 'origin', 'main')\n    if pull.returncode != 0:\n        raise RuntimeError(f'Failed to fast-forward main:\n{pull.stderr.strip()}')\n\n    updated = _run_git(repo_root, 'rev-parse', 'HEAD').stdout.strip()\n    print(f'Repository updated to latest main ({updated[:12]}). Continuing in this runtime.')\n    return repo_root\n\n\nREPO_ROOT = ensure_latest_main()\n\ntry:\n    import ipywidgets as widgets\n    from IPython.display import display\nexcept Exception:\n    widgets = None\n    display = None\n\ntry:\n    from google.colab import files as colab_files\nexcept Exception:\n    colab_files = None\n\n\n# Optional explicit overrides\nRUN_DIR = None  # e.g. Path('/content/output-dsfb-dscd/20260302_000000')\nOUTPUT_ROOT = None  # e.g. Path('/content/output-dsfb-dscd')\nRESULTS_ZIP_PATH = None  # e.g. Path('/content/dsfb-dscd-results.zip')\n\nTAU_STEPS_OPTIONS = [201, 402, 804, 1608]\nEVENT_COUNT_OPTIONS = [512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, 262144]\nEDGE_PREVIEW_OPTIONS = [128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768]\n\nDEFAULT_TAU_STEPS = 201\nDEFAULT_NUM_EVENTS = 512\nDEFAULT_EDGE_PREVIEW = 512\n\ntau_steps_dropdown = None\nnum_events_dropdown = None\nedge_preview_dropdown = None\n\nif widgets is not None and display is not None:\n    tau_steps_dropdown = widgets.Dropdown(\n        options=TAU_STEPS_OPTIONS,\n        value=DEFAULT_TAU_STEPS,\n        description='tau steps',\n    )\n    num_events_dropdown = widgets.Dropdown(\n        options=EVENT_COUNT_OPTIONS,\n        value=DEFAULT_NUM_EVENTS,\n        description='events',\n    )\n    edge_preview_dropdown = widgets.Dropdown(\n        options=EDGE_PREVIEW_OPTIONS,\n        value=DEFAULT_EDGE_PREVIEW,\n        description='edges',\n    )\n    display(widgets.HBox([num_events_dropdown, tau_steps_dropdown, edge_preview_dropdown]))\nelse:\n    print('ipywidgets unavailable; using default controls: '\n          f'events={DEFAULT_NUM_EVENTS}, tau_steps={DEFAULT_TAU_STEPS}, edges={DEFAULT_EDGE_PREVIEW}')\n\n\ndef selected_colab_params() -> dict:\n    return {\n        'num_events': int(num_events_dropdown.value) if num_events_dropdown is not None else DEFAULT_NUM_EVENTS,\n        'tau_steps': int(tau_steps_dropdown.value) if tau_steps_dropdown is not None else DEFAULT_TAU_STEPS,\n        'edge_preview_limit': int(edge_preview_dropdown.value) if edge_preview_dropdown is not None else DEFAULT_EDGE_PREVIEW,\n    }\n\n\ndef print_sweep_command_from_controls() -> None:\n    params = selected_colab_params()\n    cmd = (\n        'cargo run --release -p dsfb-dscd -- '\n        f"--num-events {params['num_events']} --tau-steps {params['tau_steps']}"\n    )\n    print('Selected controls:', params)\n    print('Suggested local generation command:')\n    print(cmd)\n\n\ndef run_sweep_from_controls() -> None:\n    params = selected_colab_params()\n    cmd = [\n        'cargo', 'run', '--release', '-p', 'dsfb-dscd', '--',\n        '--num-events', str(params['num_events']),\n        '--tau-steps', str(params['tau_steps']),\n    ]\n    print('+', ' '.join(cmd))\n    subprocess.run(cmd, check=True, cwd=REPO_ROOT)\n\n\nprint_sweep_command_from_controls()\n\nAUTO_BOOTSTRAP_SWEEP = False\n\ndef _cargo_available() -> bool:\n    return shutil.which('cargo') is not None\n\n\ndef ensure_cargo_available() -> bool:\n    if _cargo_available():\n        return True\n    if not _in_colab_runtime():\n        return False\n\n    print('cargo not found; installing Rust toolchain with rustup...')\n    install = subprocess.run(\n        ['bash', '-lc', 'curl https://sh.rustup.rs -sSf | sh -s -- -y'],\n        stdout=subprocess.PIPE,\n        stderr=subprocess.PIPE,\n        text=True,\n    )\n    if install.returncode != 0:\n        print(install.stderr.strip())\n        return False\n\n    cargo_bin = Path.home() / '.cargo' / 'bin'\n    os.environ['PATH'] = f"{cargo_bin}:{os.environ.get('PATH', '')}"\n    return _cargo_available()\n\n\ndef _is_sweep_run_dir(path: Path) -> bool:\n    return (path / 'threshold_sweep.csv').exists()\n\n\ndef _is_scaling_run_dir(path: Path) -> bool:\n    return (\n        (path / 'threshold_scaling_summary.csv').exists()\n        and any(path.glob('threshold_curve_N_*.csv'))\n    )\n\n\ndef _is_run_dir(path: Path) -> bool:\n    return path.is_dir() and (_is_sweep_run_dir(path) or _is_scaling_run_dir(path))\n\n\ndef _unique_paths(paths: List[Path]) -> List[Path]:\n    out: List[Path] = []\n    seen = set()\n    for path in paths:\n        key = str(path)\n        if key in seen:\n            continue\n        seen.add(key)\n        out.append(path)\n    return out\n\n\ndef _default_unpack_root() -> Path:\n    if Path('/content').exists():\n        return Path('/content/output-dsfb-dscd')\n    return Path.cwd().resolve() / 'output-dsfb-dscd'\n\n\ndef _candidate_output_roots() -> List[Path]:\n    roots: List[Path] = []\n\n    if OUTPUT_ROOT is not None:\n        roots.append(Path(OUTPUT_ROOT).expanduser().resolve())\n\n    cwd = Path.cwd().resolve()\n    roots.extend([\n        cwd / 'output-dsfb-dscd',\n        *[parent / 'output-dsfb-dscd' for parent in cwd.parents],\n        Path.home() / 'output-dsfb-dscd',\n        Path('/content/output-dsfb-dscd'),\n        Path('/content/dsfb/output-dsfb-dscd'),\n        Path('/content/dsfb/crates/dsfb-dscd/output-dsfb-dscd'),\n        Path('/workspace/output-dsfb-dscd'),\n        Path('/workspaces/output-dsfb-dscd'),\n        Path('/tmp/output-dsfb-dscd'),\n    ])\n\n    return _unique_paths([path.resolve() for path in roots])\n\n\ndef _find_runs_under(root: Path) -> List[Path]:\n    if not root.exists():\n        return []\n\n    if _is_run_dir(root):\n        return [root]\n\n    runs = []\n    markers = ['finite_size_scaling.csv', 'threshold_scaling_summary.csv', 'threshold_sweep.csv']\n    for marker in markers:\n        for marker_path in root.rglob(marker):\n            candidate = marker_path.parent\n            if _is_run_dir(candidate):\n                runs.append(candidate)\n    return sorted(_unique_paths(runs))\n\n\ndef _latest_run_from_roots(roots: List[Path]) -> Optional[Path]:\n    candidates: List[Path] = []\n    for root in roots:\n        candidates.extend(_find_runs_under(root))\n\n    if not candidates:\n        return None\n    return sorted(_unique_paths(candidates))[-1]\n\n\ndef _extract_zip_bytes(zip_name: str, payload: bytes, unpack_root: Path) -> Path:\n    unpack_root.mkdir(parents=True, exist_ok=True)\n    print(f'Unpacking {zip_name} into {unpack_root} ...')\n    with zipfile.ZipFile(io.BytesIO(payload), 'r') as archive:\n        archive.extractall(unpack_root)\n\n    run_dir = _latest_run_from_roots([unpack_root])\n    if run_dir is None:\n        raise FileNotFoundError(\n            f'No DSCD run folder found after unpacking {zip_name} into {unpack_root}. '\n            'Expected threshold_sweep.csv or threshold_scaling_summary.csv + threshold_curve_N_*.csv.'\n        )\n    return run_dir\n\n\ndef _extract_zip_path(zip_path: Path, unpack_root: Path) -> Path:\n    zip_path = Path(zip_path).expanduser().resolve()\n    if not zip_path.exists():\n        raise FileNotFoundError(f'Zip file not found: {zip_path}')\n\n    unpack_root.mkdir(parents=True, exist_ok=True)\n    print(f'Unpacking {zip_path} into {unpack_root} ...')\n    with zipfile.ZipFile(zip_path, 'r') as archive:\n        archive.extractall(unpack_root)\n\n    run_dir = _latest_run_from_roots([unpack_root])\n    if run_dir is None:\n        raise FileNotFoundError(\n            f'No DSCD run folder found after unpacking {zip_path} into {unpack_root}. '\n            'Expected threshold_sweep.csv or threshold_scaling_summary.csv + threshold_curve_N_*.csv.'\n        )\n    return run_dir\n\n\ndef upload_and_unpack_results_zip() -> Path:\n    if colab_files is None:\n        raise RuntimeError('Google Colab file picker is unavailable in this environment.')\n\n    print('Pick a DSCD results .zip from your local machine...')\n    uploaded = colab_files.upload()\n    if not uploaded:\n        raise RuntimeError('No file uploaded.')\n\n    zip_name, payload = next(iter(uploaded.items()))\n    if not zip_name.lower().endswith('.zip'):\n        raise ValueError(f'Uploaded file is not a .zip: {zip_name}')\n\n    return _extract_zip_bytes(zip_name, payload, _default_unpack_root())\n\n\n\n\ndef _ensure_finite_size_scaling_csv(run_dir: Path) -> None:\n    target = run_dir / 'finite_size_scaling.csv'\n    if target.exists():\n        return\n\n    summary_path = run_dir / 'threshold_scaling_summary.csv'\n    if not summary_path.exists():\n        return\n\n    rows = []\n    with summary_path.open('r', newline='') as fh:\n        reader = csv.DictReader(fh)\n        for row in reader:\n            num_events = row.get('num_events') or row.get('N')\n            width = row.get('transition_width') or row.get('width_0_1_to_0_9')\n            max_derivative = row.get('max_derivative') or '0'\n            if not num_events or width is None:\n                continue\n            rows.append({\n                'num_events': num_events,\n                'transition_width': width,\n                'width_0_1_to_0_9': width,\n                'max_derivative': max_derivative,\n            })\n\n    if not rows:\n        return\n\n    with target.open('w', newline='') as fh:\n        writer = csv.DictWriter(\n            fh,\n            fieldnames=['num_events', 'transition_width', 'width_0_1_to_0_9', 'max_derivative'],\n        )\n        writer.writeheader()\n        writer.writerows(rows)\n\n    print(f'Generated missing {target.name} from threshold_scaling_summary.csv')\n\n\ndef render_zip_upload_widget() -> None:\n    if widgets is None or display is None:\n        print('ipywidgets unavailable; use RESULTS_ZIP_PATH or upload_and_unpack_results_zip() in Colab.')\n        return\n\n    uploader = widgets.FileUpload(accept='.zip', multiple=False, description='Upload DSCD zip')\n    out = widgets.Output()\n\n    def _on_change(change):\n        value = uploader.value\n        if not value:\n            return\n\n        try:\n            if isinstance(value, dict):\n                item = next(iter(value.values()))\n                name = item.get('name', 'uploaded.zip')\n                content = item.get('content', b'')\n            else:\n                item = value[0]\n                name = item.get('name', 'uploaded.zip')\n                content = item.get('content', b'')\n\n            run_dir = _extract_zip_bytes(name, bytes(content), _default_unpack_root())\n            with out:\n                out.clear_output()\n                print(f'Ready: {run_dir}')\n        except Exception as exc:\n            with out:\n                out.clear_output()\n                print(f'Upload failed: {exc}')\n\n    uploader.observe(_on_change, names='value')\n    display(uploader, out)\n\n\n# Show zip picker early on Colab before any auto-generation path.\nif _in_colab_runtime() and RUN_DIR is None and RESULTS_ZIP_PATH is None:\n    print('Upload a DSCD results .zip (preferred on free Colab):')\n    render_zip_upload_widget()\n\n\nresolved_run_dir: Optional[Path] = None\n\nif RUN_DIR is not None:\n    run_dir = Path(RUN_DIR).expanduser().resolve()\n    if not _is_run_dir(run_dir):\n        raise FileNotFoundError(\n            f'Configured RUN_DIR is not a valid DSCD run folder: {run_dir}. '\n            'Expected threshold_sweep.csv or threshold_scaling_summary.csv + threshold_curve_N_*.csv.'\n        )\n    resolved_run_dir = run_dir\nelse:\n    resolved_run_dir = _latest_run_from_roots(_candidate_output_roots())\n\nif resolved_run_dir is None and RESULTS_ZIP_PATH is not None:\n    resolved_run_dir = _extract_zip_path(Path(RESULTS_ZIP_PATH), _default_unpack_root())\n\nif resolved_run_dir is None and colab_files is not None:\n    print('No local DSCD run found. Launching Colab zip picker...')\n    resolved_run_dir = upload_and_unpack_results_zip()\n\nif resolved_run_dir is None and RESULTS_ZIP_PATH is None and AUTO_BOOTSTRAP_SWEEP:\n    if ensure_cargo_available():\n        print('No local DSCD run found. Auto-generating from selected dropdown controls...')\n        run_sweep_from_controls()\n        resolved_run_dir = _latest_run_from_roots(_candidate_output_roots())\n    else:\n        print('Cargo unavailable; cannot auto-generate DSCD run in this runtime.')\n\nif resolved_run_dir is None:\n    searched = '\n'.join(f' - {path}' for path in _candidate_output_roots())\n    print('No DSCD outputs found yet.')\n    print('Options:')\n    print('1) Set RUN_DIR to a run folder with threshold_sweep.csv or threshold_scaling_summary.csv + threshold_curve_N_*.csv')\n    print('2) Set RESULTS_ZIP_PATH to a local .zip path and rerun this cell')\n    print('3) In Colab, call upload_and_unpack_results_zip()')\n    print('Searched:\n' + searched)\n    if colab_files is None:\n        render_zip_upload_widget()\n    raise FileNotFoundError('No DSCD output folder available for replay.')\n\n_ensure_finite_size_scaling_csv(resolved_run_dir)\n\nRUN_DIR = resolved_run_dir\nOUTPUT_ROOT = RUN_DIR.parent\nprint(f'Using OUTPUT_ROOT: {OUTPUT_ROOT}')\nprint(f'Using RUN_DIR: {RUN_DIR}')\nRUN_DIR\n

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

sweep_path = RUN_DIR / 'threshold_sweep.csv'
if sweep_path.exists():
    threshold_df = pd.read_csv(sweep_path)
    print(f'Using sweep file: {sweep_path.name}')
else:
    curve_paths = sorted(RUN_DIR.glob('threshold_curve_N_*.csv'))
    if not curve_paths:
        raise FileNotFoundError(
            'Missing threshold_sweep.csv and no threshold_curve_N_*.csv files found in RUN_DIR'
        )

    def _curve_n(path: Path) -> int:
        try:
            return int(path.stem.split('_')[-1])
        except Exception:
            return -1

    selected_curve = max(curve_paths, key=_curve_n)
    threshold_df = pd.read_csv(selected_curve)
    print(f'Using scaling curve file: {selected_curve.name} (no threshold_sweep.csv present)')

threshold_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(threshold_df['tau'], threshold_df['expansion_ratio'], linewidth=2)
ax.set_title('DSCD expansion ratio vs trust threshold')
ax.set_xlabel('tau')
ax.set_ylabel('expansion_ratio')
ax.grid(alpha=0.25)
plt.show()

In [ ]:
finite_df = pd.read_csv(RUN_DIR / 'finite_size_scaling.csv')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(finite_df['num_events'], finite_df['transition_width'], marker='o')
axes[0].set_title('Transition width vs N')
axes[0].set_xlabel('num_events')
axes[0].set_ylabel('transition_width')
axes[0].grid(alpha=0.25)

axes[1].plot(finite_df['num_events'], finite_df['max_derivative'], marker='o')
axes[1].set_title('Max derivative vs N')
axes[1].set_xlabel('num_events')
axes[1].set_ylabel('max_derivative')
axes[1].grid(alpha=0.25)
plt.show()


In [ ]:
events_path = RUN_DIR / 'graph_events.csv'
edges_path = RUN_DIR / 'graph_edges.csv'

events_df = pd.read_csv(events_path)
edges_df = pd.read_csv(edges_path)

params = selected_colab_params()
edge_preview_limit = params['edge_preview_limit']
if 'source_event_id' in edges_df.columns and 'target_event_id' in edges_df.columns:
    src_col, dst_col = 'source_event_id', 'target_event_id'
else:
    src_col, dst_col = 'from_event_id', 'to_event_id'

subset_edges = edges_df.head(edge_preview_limit)
subset_nodes = sorted(set(subset_edges[src_col]).union(subset_edges[dst_col]))

graph = nx.DiGraph()
graph.add_nodes_from(subset_nodes)
graph.add_edges_from(subset_edges[[src_col, dst_col]].itertuples(index=False, name=None))

plt.figure(figsize=(10, 6))
pos = nx.spring_layout(graph, seed=7)
nx.draw_networkx(graph, pos=pos, node_size=120, with_labels=False, arrows=True)
plt.title(f'DSCD graph preview (first {len(subset_edges)} edges)')
plt.axis('off')
plt.show()